In [21]:
!pip install -q google-genai chromadb pypdf
!pip install -q -U fsspec huggingface_hub
!pip install -q sentence-transformers
!pip install -q streamlit pyngrok


In [22]:
import os, textwrap, chromadb
from pypdf import PdfReader
#from sentence_transformers import SentenceTransformer
from google import genai
from google.colab import drive, files

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
from google.colab import userdata
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

def ask_gemini(question):
    try:
        response = client.models.generate_content(
          model="gemini-3.5-flash",
          contents=question
          )
        return response.text
    except Exception as e:
        return f"Error: {e}"


In [ ]:
chat_history = []

def ask_with_memory(prompt):
    chat_history.append(f"User: {prompt}")
    conversation = "\n".join(chat_history)
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=conversation)
    answer = response.text
    chat_history.append(f"Assistant: {answer}")
    return answer


In [24]:
def read_pdf():
    uploaded = files.upload()
    text = ""
    for file_name in uploaded.keys():
        reader = PdfReader(Python_Notes.pdf)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

pdf_text = ""


In [ ]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

def create_chunks(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks


In [ ]:
!pip install -q sentence-transformers
!pip install -q -U fsspec huggingface_hub
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
client_db = chromadb.Client()
collection = client_db.get_or_create_collection(name="study_notes")

def index_pdf(text_to_index):
    global chunks, embeddings
    chunks = create_chunks(text_to_index)
    embeddings = embedding_model.encode(chunks)
    collection.add(
        documents=chunks,
        embeddings=embeddings.tolist(),
        ids=[str(i) for i in range(len(chunks))]
    )


In [ ]:
def rag_answer(question):
    query_embedding = embedding_model.encode(question)
    results = collection.query(
        query_embeddings=[query_embedding.tolist()], n_results=3
    )

    retrieved_docs = (
        results["documents"][0]
        if (results.get("documents") and results["documents"][0])
        else []
    )

    distances = (
        results["distances"][0]
        if (results.get("distances") and results["distances"][0])
        else []
    )
    is_relevant = len(retrieved_docs) > 0 and (
        len(distances) == 0 or distances[0] < 1.3
    )

    if is_relevant:
        context = "\n\n".join(retrieved_docs)
        prompt = f"""You are a helpful assistant. Answer the user's question primarily using the provided document context. If the context does not fully answer the question, supplement it with your general knowledge, but prioritize the document information.

Document Context:
{context}

Question:
{question}"""
    else:
        prompt = question

    return ask_with_memory(prompt)

In [ ]:
def generate_quiz(topic):
    query_embedding = embedding_model.encode(topic)
    results = collection.query(query_embeddings=[query_embedding.tolist()], n_results=5)
    context = "\n\n".join(results["documents"][0])
    prompt = f"Using only the context below, generate 5 multiple-choice questions.\nContext:\n{context}\nFormat:\nQuestion\nA)\nB)\nC)\nD)\nAnswer:"
    response = client.models.generate_content(model="gemini-3.5-flash", contents=prompt)
    return response.text


In [ ]:

def generate_study_plan(days):
    query = "Complete syllabus and important topics"
    query_embedding = embedding_model.encode(query)
    results = collection.query(query_embeddings=[query_embedding.tolist()], n_results=5)
    context = "\n\n".join(results["documents"][0])
    prompt = f"You are an expert study planner.\nUsing only the context below, create a {days}-day study plan.\nEach day should include:\n- Topics to study\n- Revision\n- Practice Questions\nContext:\n{context}"
    response = client.models.generate_content(model="gemini-3.5-flash", contents=prompt)
    return response.text


In [ ]:
def generate_project_idea(topic):
    query_embedding = embedding_model.encode(topic)
    results = collection.query(query_embeddings=[query_embedding.tolist()], n_results=5)
    context = "\n\n".join(results["documents"][0])
    prompt = f"You are a Project Mentor.\nUsing the context below, design a practical, hands-on project based on the subject matter.\nInclude:\n- Project Title & Objective\n- Step-by-Step Implementation Guide\n- Expected Deliverables\nContext:\n{context}\nTopic/Interest:\n{topic}"
    response = client.models.generate_content(model="gemini-3.5-flash", contents=prompt)
    return response.text


In [ ]:
def generate_career_guidance(query):
    query_embedding = embedding_model.encode(query)
    results = collection.query(query_embeddings=[query_embedding.tolist()], n_results=5)
    context = "\n\n".join(results["documents"][0])
    prompt = f"You are an Industry Career Counselor.\nUsing the provided course context, outline potential career paths and relevant industry roles.\nInclude:\n- Target Job Roles\n- Key Industry Skills Demonstrated\n- Recommended Next Steps / Resume Bullets\nContext:\n{context}\nUser Query:\n{query}"
    response = client.models.generate_content(model="gemini-3.5-flash", contents=prompt)
    return response.text


In [ ]:
def router(prompt):
    text = prompt.lower()
    if "quiz" in text:
        return "QUIZ"
    elif "study plan" in text or "schedule" in text:
        return "PLANNER"
    elif "project" in text or "hands-on" in text or "assignment" in text:
        return "PROJECT"
    elif "career" in text or "job" in text or "interview" in text or "resume" in text:
        return "CAREER"
    elif "pdf" in text or "upload" in text:
        return "PDF"
    else:
        return "CHAT"

while True:
    question = input("You: ")
    if question.lower() == "exit":
        break
    try:
        route = router(question)
        if route == "CHAT":
            print("\nGemini:", rag_answer(question))
        elif route == "QUIZ":
            print("\n", generate_quiz(question))
        elif route == "PLANNER":
            days = input("How many study days? ")
            print("\n", generate_study_plan(days))
        elif route == "PROJECT":
            print("\n", generate_project_idea(question))
        elif route == "CAREER":
            print("\n", generate_career_guidance(question))
        elif route == "PDF":
            pdf_text = read_pdf()
            index_pdf(pdf_text)
            print("\nPDF Loaded and Indexed Successfully.")
        print()
    except Exception as e:
        print("\nError:", e)
        print()


You: Exit


In [27]:
import streamlit as st

st.set_page_config(page_title="Agentic AI Study Assistant", page_icon="📚")
st.title("📚 Agentic AI Study Assistant")

st.sidebar.header("Navigation")
st.write("Welcome! Upload your study material or ask a question to get started.")

# Add your notebook's agent invocation logic here


2026-07-25 19:11:09.865 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-25 19:11:09.867 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-25 19:11:09.869 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-25 19:11:09.871 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-25 19:11:09.872 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-25 19:11:09.873 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-25 19:11:09.874 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-25 19:11:09.875 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [33]:
import os

# 1. Ensure folder exists
os.makedirs("Agentic_AI_Study_Assistant", exist_ok=True)

# 2. Define app.py content
app_code = """import streamlit as st

st.set_page_config(page_title="Agentic AI Study Assistant", layout="wide")
st.title("Agentic AI Study Assistant 📚")

st.write("Hello from app.py inside Google Colab!")
"""

# 3. Write file
file_path = "Agentic_AI_Study_Assistant/app.py"
with open(file_path, "w") as f:
    f.write(app_code)

print(f"✅ File successfully created at: {file_path}")


✅ File successfully created at: Agentic_AI_Study_Assistant/app.py


In [44]:
%%writefile app.py
import streamlit as st
st.title("Agentic AI Study Assistant")
st.write("Welcome!")


Overwriting app.py


In [48]:
%%writefile README.md

Agentic AI Study Assistant
This project uses Gemini + RAG + ChromaDB.

Overwriting README.md


In [45]:
%%writefile requirements.txt

streamlit
google-genai
chromadb
sentence-transformers
pypdf
pyngrok
streamlit
google-genai
chromadb
sentence-transformers
pypdf
pyngrok

Overwriting requirements.txt


In [39]:
%%writefile .gitignore

__pycache__/
.env
*.pyc
chroma_db/

Writing .gitignore
